# Notebook 48R2 - Binary Ensemble Test
Tests whether binary ensemble strategies outperform greyscale blending.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import cv2, numpy as np, pandas as pd, json
from pathlib import Path
from skimage.metrics import structural_similarity as ssim

In [ ]:
ROOT=Path("/content/drive/MyDrive/MusicAnnotationProject")
N46=ROOT/"observed_print_classifier_v8_6_notebook46"
RUN46=max([p for p in N46.iterdir() if p.is_dir()],key=lambda p:p.stat().st_mtime)
OUT=RUN46/"notebook48R2_binary_ensemble"
HYB=OUT/"candidates"
TAB=OUT/"tables"
for p in [OUT,HYB,TAB]: p.mkdir(parents=True,exist_ok=True)
inv=pd.read_csv(RUN46/"tables"/"whole_page_candidate_inventory.csv")

In [ ]:
def ink(im):
 return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def f1_from_features(ref,pred,reg,tol=1):
 import numpy as np, cv2
 ref=ref&reg; pred=pred&reg
 if not ref.any(): return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8)
 p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
 q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
 return float(p),float(q),float(2*p*q/max(p+q,1e-12))

In [ ]:
pairs=inv[inv.restorer.isin(["R1_biharmonic","R3_learned_unet"])]
pairs=pairs.pivot_table(index=["page","source_group","arm"],columns="restorer",values="candidate",aggfunc="first").reset_index()
rows=[]
for _,r in pairs.iterrows():
 b=cv2.imread(str(r["R1_biharmonic"]),0)
 u=cv2.imread(str(r["R3_learned_unet"]),0)
 h50=((b.astype(float)+u.astype(float))/2).astype(np.uint8)
 _,otsu=cv2.threshold(h50,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
 and_img=((ink(b)&ink(u))*255).astype(np.uint8)
 or_img=((ink(b)|ink(u))*255).astype(np.uint8)
 products={"Hybrid50":h50,"Hybrid50_Otsu":otsu,"AND_Ensemble":and_img,"OR_Ensemble":or_img}
 for name,img in products.items():
  fp=HYB/f"{r.page}_{r.arm}_{name}.png"
  cv2.imwrite(str(fp),img)
  rows.append({"page":r.page,"source_group":r.source_group,"arm":r.arm,"restorer":name,"candidate":str(fp)})
ensemble_inventory=pd.DataFrame(rows)
ensemble_inventory.to_csv(TAB/"ensemble_inventory.csv",index=False)

In [ ]:
# Copy the page_clean/page_obs/page_auth reconstruction cells from 46R6 here.
# Then replace whole_candidates with ensemble_inventory and run the same whole-page scoring block.
print("Generated ensemble inventory:", len(ensemble_inventory))